# PR #1305: Cache Point Bug Demo

This notebook demonstrates the bug that PR #1305 fixes:
- **Working**: System blocks with only standard fields (`text`, `cachePoint`)
- **Bug**: System blocks with extra fields cause `ParamValidationError`

In [ ]:
from strands import Agent
from strands.models import BedrockModel
import uuid

In [ ]:
# Create a long system prompt (>1000 tokens required for Nova caching)
unique_id = str(uuid.uuid4())[:8]

base_prompt = """
You are a customer support assistant with expertise in helping users.

Your responsibilities:
- Answer product questions clearly and accurately
- Help troubleshoot technical issues step by step
- Process returns and refunds according to policy
- Escalate complex issues to human agents when needed

Guidelines:
- Be friendly, patient, and professional
- Ask clarifying questions when needed
- Provide accurate information only
- Respect customer privacy
""" * 20  # Repeat to exceed 1000 tokens

system_prompt_text = f"[Session: {unique_id}]\n\n{base_prompt}"
print(f"Session: {unique_id}")
print(f"Length: ~{len(system_prompt_text)//4} tokens")

---
## Working: Clean Content Blocks

System blocks with only `text` and `cachePoint` fields work correctly.

In [ ]:
# Clean content blocks - only standard fields
system_prompt_clean = [
    {"text": system_prompt_text},
    {"cachePoint": {"type": "default"}},
]

model = BedrockModel(model_id="amazon.nova-lite-v1:0", region_name="us-east-1")
agent = Agent(model=model, system_prompt=system_prompt_clean)

In [ ]:
# First request - cache WRITE
result1 = agent("Hello!")

print(f"Response: {str(result1)[:150]}...")
print(f"cacheWriteInputTokens: {result1.metrics.accumulated_usage.get('cacheWriteInputTokens', 0)}")
print(f"cacheReadInputTokens: {result1.metrics.accumulated_usage.get('cacheReadInputTokens', 0)}")

In [ ]:
# Second request - cache READ
result2 = agent("What can you help me with?")

print(f"Response: {str(result2)[:150]}...")
print(f"cacheWriteInputTokens: {result2.metrics.accumulated_usage.get('cacheWriteInputTokens', 0)}")
print(f"cacheReadInputTokens: {result2.metrics.accumulated_usage.get('cacheReadInputTokens', 0)}")

if result2.metrics.accumulated_usage.get('cacheReadInputTokens', 0) > 0:
    print("\nCache HIT")

---
## Bug: Extra Fields in Content Blocks

When content blocks have extra fields, Bedrock rejects them with `ParamValidationError`.

**This is the bug PR #1305 fixes.**

In [ ]:
# Content blocks with EXTRA fields - causes error
unique_id_error = str(uuid.uuid4())[:8]
system_prompt_error_text = f"[Session: {unique_id_error}]\n\n{base_prompt}"

system_prompt_with_extra = [
    {
        "text": system_prompt_error_text,
        "extraField": "metadata",    # NOT allowed by Bedrock
        "source": "config",          # NOT allowed by Bedrock
    },
    {"cachePoint": {"type": "default"}},
]

print("Block 0 keys:", list(system_prompt_with_extra[0].keys()))
print("Extra fields will cause ParamValidationError")

In [ ]:
# This FAILS - demonstrates the bug
try:
    model_error = BedrockModel(model_id="amazon.nova-lite-v1:0", region_name="us-east-1")
    agent_error = Agent(model=model_error, system_prompt=system_prompt_with_extra)
    result_error = agent_error("Hello!")
except Exception as e:
    print(f"Error: {type(e).__name__}")
    print(f"\n{e}")

---
## The Fix (PR #1305)

**Problem**: Messages are cleaned via `_format_bedrock_messages()`, but system blocks are passed directly without sanitization.

**Solution**: PR #1305 adds sanitization for system blocks, stripping extra fields before sending to Bedrock.

| Scenario | Result |
|----------|--------|
| `{"text": "..."}` | Works |
| `{"text": "...", "extra": "..."}` | Fails (fixed by PR #1305) |